# Proyecto 6 - Stable Diffusion y generación condicionada por texto

**Línea de proyecto:** Modelos de difusión, DDPM/DDIM, Stable Diffusion y condicionamiento<br>
**Material base:** `Cuaderno25-CC0C2.ipynb` (difusión, latentes, guidance)

## Objetivo

Usar un pipeline de **Stable Diffusion** (o variante ligera) para generar imágenes a partir de prompts, mostrar el flujo **tokenización -> encoding textual -> difusión en latente -> decodificación**, y comparar dos valores de **guidance scale** analizando el trade-off entre fidelidad al prompt y diversidad visual.

## Trazabilidad

| Componente | Origen |
|---|---|
| Conceptos latentes, CFG, scheduler | `Cuaderno25-CC0C2.ipynb` |
| Pipeline diffusers y comparación guidance | Implementación propia |
| README / guía video | IA para estructura; interpretación propia |

## Importaciones

In [1]:
import os
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

## Configuración experimental

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_ID = "segmind/tiny-sd"
GUIDANCE_A = 3.5
GUIDANCE_B = 9.0
NUM_INFERENCE_STEPS = 20
IMAGE_SIZE = 512
PROMPT = "Un laboratorio de inteligencia artificial futurista, estilo infografía técnica, colores azul y blanco"
NEGATIVE_PROMPT = "borroso, deforme, texto ilegible, baja calidad"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

print("Dispositivo:", device)
print("Modelo:", MODEL_ID)
print("Semilla:", SEED)
print("Guidance A/B:", GUIDANCE_A, GUIDANCE_B)
print("Pasos de inferencia:", NUM_INFERENCE_STEPS)

Dispositivo: cpu
Modelo: segmind/tiny-sd
Semilla: 42
Guidance A/B: 3.5 9.0
Pasos de inferencia: 20


## Celda de verificación personal

In [3]:
STUDENT_NAME = "Rojas Huaroc, Luis Antonio"
EXECUTION_DATE = "2026-07-04"
PROJECT_TRACK = "Proyecto 6: Stable Diffusion y generación condicionada"
MODEL_NAME = MODEL_ID
VARIANT = f"Comparación guidance_scale {GUIDANCE_A} vs {GUIDANCE_B}"

print("Estudiante:", STUDENT_NAME)
print("Fecha:", EXECUTION_DATE)
print("Línea de proyecto:", PROJECT_TRACK)
print("Modelo:", MODEL_NAME)
print("Semilla:", SEED)
print("Variante:", VARIANT)
print("Frase técnica propia:", "El guidance scale interpola entre la predicción condicionada y no condicionada para controlar obediencia al prompt sin reentrenar el UNet.")


Estudiante: Rojas Huaroc, Luis Antonio
Fecha: 2026-07-04
Línea de proyecto: Proyecto 6: Stable Diffusion y generación condicionada
Modelo: segmind/tiny-sd
Semilla: 42
Variante: Comparación guidance_scale 3.5 vs 9.0
Frase técnica propia: El guidance scale interpola entre la predicción condicionada y no condicionada para controlar obediencia al prompt sin reentrenar el UNet.


## Carga del pipeline Stable Diffusion

In [4]:
from diffusers import StableDiffusionPipeline, DDIMScheduler

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
    use_safetensors=False,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

if device.type == "cpu":
    pipe.enable_attention_slicing()

print("Componentes del pipeline:", list(pipe.components.keys()))
print("Scheduler:", pipe.scheduler.__class__.__name__)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--segmind--tiny-sd/snapshots/cad0bd7495fa6c4bcca01b19a723dc91627fe84f/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--segmind--tiny-sd/snapshots/cad0bd7495fa6c4bcca01b19a723dc91627fe84f/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--segmind--tiny-sd/snapshots/cad0bd7495fa6c4bcca01b19a723dc91627fe84f/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--segmind--tiny-sd/snapshots/cad0bd7495fa6c4bcca01b19a723dc91627fe84f/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Componentes del pipeline: ['vae', 'text_encoder', 'tokenizer', 'unet', 'scheduler', 'safety_checker', 'feature_extractor', 'image_encoder']
Scheduler: DDIMScheduler
